In [ ]:

import sys
import os
import tifffile
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
sys.path.append('/home/cyf/wbi/Virginia/code_for_paper/wholistic_registration/src/wholistic_registration')
current_dir = os.path.dirname(os.path.abspath("__file__"))
root_dir = os.path.dirname(current_dir)
sys.path.append(root_dir)

root_dir = Path('/home/cyf/wbi/Virginia/code_for_paper/data/simulation_v0420/repos')
from utils import simulation
# root_dir = Path('/home/cyf/wbi/Virginia/code/src/wholistic_registration/exp/simulation/repos')
zslices = [2, 10, 18]  # 选择 z slice
experiment_groups = ['art_R', 'amp', 'noise_level']  # 对应三组实验
results = {}  # 保存 MSE
results = {group: {} for group in experiment_groups}
repo_dirs = sorted([d for d in root_dir.iterdir() if d.is_dir() and d.name.startswith('rep')])
num_repos = len(repo_dirs)
print(f"Found {num_repos} repos: {[d.name for d in repo_dirs]}")

In [1]:
import numpy as np

def _safe_ncc(a, b, eps=1e-6):
    """
    Normalized cross-correlation between two patches.
    a, b: same shape
    """
    a = a.astype(np.float32, copy=False)
    b = b.astype(np.float32, copy=False)

    a = a - np.mean(a)
    b = b - np.mean(b)

    denom = np.sqrt(np.sum(a * a) * np.sum(b * b)) + eps
    return float(np.sum(a * b) / denom)

def _softmax_np(x, temperature=0.1, axis=-1):
    """
    Stable softmax.
    Smaller temperature makes distribution sharper.
    """
    x = np.asarray(x, dtype=np.float32)
    x = x / float(temperature)
    x = x - np.max(x, axis=axis, keepdims=True)
    ex = np.exp(x)
    return ex / (np.sum(ex, axis=axis, keepdims=True) + 1e-12)

def _make_disp_candidates(search_radius, search_step=(1, 1, 1)):
    """
    Generate displacement candidates in [dy, dx, dz] convention.

    search_radius: int or tuple
        If int: same radius for y/x/z.
        If tuple: (ry, rx, rz)
    search_step: tuple
        (sy, sx, sz)
    """
    if isinstance(search_radius, int):
        ry = rx = rz = search_radius
    else:
        ry, rx, rz = search_radius

    sy, sx, sz = search_step

    dys = np.arange(-ry, ry + 1, sy, dtype=np.int32)
    dxs = np.arange(-rx, rx + 1, sx, dtype=np.int32)
    dzs = np.arange(-rz, rz + 1, sz, dtype=np.int32)

    candidates = []
    for dy in dys:
        for dx in dxs:
            for dz in dzs:
                candidates.append([dy, dx, dz])

    return np.asarray(candidates, dtype=np.int32)

def _valid_patch_bounds(center, half_size, shape):
    """
    Check whether patch centered at center is valid.
    center: (cy, cx, cz)
    half_size: (hy, hx, hz)
    shape: (Y, X, Z)
    """
    cy, cx, cz = center
    hy, hx, hz = half_size
    Y, X, Z = shape

    return (
        cy - hy >= 0 and cy + hy + 1 <= Y and
        cx - hx >= 0 and cx + hx + 1 <= X and
        cz - hz >= 0 and cz + hz + 1 <= Z
    )

def _extract_patch(vol, center, half_size):
    """
    Extract patch from volume.
    vol: (Y, X, Z)
    center: (cy, cx, cz)
    half_size: (hy, hx, hz)
    """
    cy, cx, cz = center
    hy, hx, hz = half_size

    return vol[
        cy - hy: cy + hy + 1,
        cx - hx: cx + hx + 1,
        cz - hz: cz + hz + 1
    ]

def run_patch_correlation_matching(
    fixed_np,
    moving_np,
    patch_size=(33, 33, 3),
    stride=(16, 16, 1),
    search_radius=(16, 16, 0),
    search_step=(1, 1, 1),
    mode="best",                 # "best" or "expectation"
    temperature=0.1,
    min_valid_score=-1.0,
    return_score_volume=True,
    verbose=False,
):
    """
    Patch-level correlation coarse matching.

    Convention
    ----------
    For each patch center p in moving_np, this function searches candidate
    positions p + d in fixed_np.

    Therefore, the output displacement d means:

        moving patch at p corresponds to fixed patch at p + d

    The displacement channel order is [dy, dx, dz], matching your usual
    [y, x, z] convention.

    Parameters
    ----------
    fixed_np : np.ndarray
        Reference/fixed volume, shape (Y, X, Z).
    moving_np : np.ndarray
        Moving volume, shape (Y, X, Z).
    patch_size : tuple
        Patch size in (py, px, pz). Each value should preferably be odd.
    stride : tuple
        Patch grid stride in (sy, sx, sz).
    search_radius : tuple
        Candidate displacement radius in (ry, rx, rz).
        Example:
            (16, 16, 0) means search xy only.
    search_step : tuple
        Candidate displacement step in (dy_step, dx_step, dz_step).
    mode : str
        "best":
            Use argmax NCC candidate.
        "expectation":
            Softmax over NCC scores, then compute expected displacement.
    temperature : float
        Softmax temperature for expectation mode.
    min_valid_score : float
        Fill invalid candidates with this score.
    return_score_volume : bool
        Whether to return score and probability volumes.
    verbose : bool

    Returns
    -------
    result : dict
        {
            "motion_patch": np.ndarray, shape (Ny, Nx, Nz, 3),
            "patch_centers": np.ndarray, shape (Ny, Nx, Nz, 3),
            "disp_candidates": np.ndarray, shape (D, 3),
            "score_volume": np.ndarray or None, shape (Ny, Nx, Nz, D),
            "prob_volume": np.ndarray or None, shape (Ny, Nx, Nz, D),
            "best_index": np.ndarray, shape (Ny, Nx, Nz),
            "valid_mask": np.ndarray, shape (Ny, Nx, Nz)
        }
    """
    fixed_np = np.asarray(fixed_np, dtype=np.float32)
    moving_np = np.asarray(moving_np, dtype=np.float32)

    assert fixed_np.shape == moving_np.shape, "fixed_np and moving_np must have same shape for this baseline."
    assert fixed_np.ndim == 3, f"Expected 3D input, got ndim={fixed_np.ndim}"

    Y, X, Z = fixed_np.shape

    py, px, pz = patch_size
    sy, sx, sz = stride

    hy, hx, hz = py // 2, px // 2, pz // 2
    half_size = (hy, hx, hz)

    disp_candidates = _make_disp_candidates(
        search_radius=search_radius,
        search_step=search_step
    )
    num_disp = len(disp_candidates)

    # Patch centers where the moving patch itself is valid.
    y_centers = np.arange(hy, Y - hy, sy, dtype=np.int32)
    x_centers = np.arange(hx, X - hx, sx, dtype=np.int32)
    z_centers = np.arange(hz, Z - hz, sz, dtype=np.int32)

    Ny, Nx, Nz = len(y_centers), len(x_centers), len(z_centers)

    motion_patch = np.full((Ny, Nx, Nz, 3), np.nan, dtype=np.float32)
    patch_centers = np.zeros((Ny, Nx, Nz, 3), dtype=np.int32)
    best_index = np.full((Ny, Nx, Nz), -1, dtype=np.int32)
    valid_mask = np.zeros((Ny, Nx, Nz), dtype=bool)

    if return_score_volume:
        score_volume = np.full((Ny, Nx, Nz, num_disp), min_valid_score, dtype=np.float32)
    else:
        score_volume = None

    if verbose:
        print(f"[Patch Corr] fixed/moving shape = {fixed_np.shape}")
        print(f"[Patch Corr] patch grid = {(Ny, Nx, Nz)}, num_disp = {num_disp}")

    for iy, cy in enumerate(y_centers):
        for ix, cx in enumerate(x_centers):
            for iz, cz in enumerate(z_centers):
                center_mov = (int(cy), int(cx), int(cz))
                patch_centers[iy, ix, iz] = center_mov

                if not _valid_patch_bounds(center_mov, half_size, moving_np.shape):
                    continue

                mov_patch = _extract_patch(moving_np, center_mov, half_size)

                scores = np.full((num_disp,), min_valid_score, dtype=np.float32)
                candidate_valid = np.zeros((num_disp,), dtype=bool)

                for idisp, d in enumerate(disp_candidates):
                    dy, dx, dz = int(d[0]), int(d[1]), int(d[2])
                    center_fix = (center_mov[0] + dy, center_mov[1] + dx, center_mov[2] + dz)

                    if not _valid_patch_bounds(center_fix, half_size, fixed_np.shape):
                        continue

                    fix_patch = _extract_patch(fixed_np, center_fix, half_size)
                    scores[idisp] = _safe_ncc(mov_patch, fix_patch)
                    candidate_valid[idisp] = True

                if not np.any(candidate_valid):
                    continue

                valid_mask[iy, ix, iz] = True

                if return_score_volume:
                    score_volume[iy, ix, iz, :] = scores

                if mode == "best":
                    idx_best = int(np.argmax(scores))
                    best_index[iy, ix, iz] = idx_best
                    motion_patch[iy, ix, iz, :] = disp_candidates[idx_best].astype(np.float32)

                elif mode == "expectation":
                    # Make invalid candidates very unlikely.
                    scores_for_softmax = scores.copy()
                    scores_for_softmax[~candidate_valid] = -1e6

                    prob = _softmax_np(scores_for_softmax, temperature=temperature, axis=0)
                    expected_disp = np.sum(prob[:, None] * disp_candidates.astype(np.float32), axis=0)

                    idx_best = int(np.argmax(scores))
                    best_index[iy, ix, iz] = idx_best
                    motion_patch[iy, ix, iz, :] = expected_disp.astype(np.float32)

                else:
                    raise ValueError(f"Unknown mode: {mode}")

    prob_volume = None
    if return_score_volume:
        # Convert score volume to probability volume.
        # For invalid patches, keep all probability as nan.
        prob_volume = np.full_like(score_volume, np.nan, dtype=np.float32)

        for iy in range(Ny):
            for ix in range(Nx):
                for iz in range(Nz):
                    if not valid_mask[iy, ix, iz]:
                        continue

                    scores = score_volume[iy, ix, iz, :]
                    prob = _softmax_np(scores, temperature=temperature, axis=0)
                    prob_volume[iy, ix, iz, :] = prob.astype(np.float32)

    result = {
        "motion_patch": motion_patch,
        "patch_centers": patch_centers,
        "disp_candidates": disp_candidates,
        "score_volume": score_volume,
        "prob_volume": prob_volume,
        "best_index": best_index,
        "valid_mask": valid_mask,
        "patch_size": patch_size,
        "stride": stride,
        "search_radius": search_radius,
        "search_step": search_step,
        "mode": mode,
        "temperature": temperature,
    }

    return result